<a href="https://colab.research.google.com/github/ALiao18/SUDC/blob/main/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
from sklearn.impute import SimpleImputer

In [ ]:
RAW_DATA_PATH = # define raw data path here
SAVE_PATH     = # define save path here

df_raw = pd.read_excel(RAW_DATA_PATH)

In [ ]:
# IDs were excluded, some may overlap, final excluded IDs = 13 cases.
EXCLUDE_IDS = [
    # 7 cases for Epilepsy / afebrile seizure
    # 7 cases Unknown sleep position
    # 5 cases Unknown sleep position (cont.)
]

df_raw = df_raw[
    (~df_raw['sudcrrc_id'].isin(EXCLUDE_IDS)) &
    (df_raw['sudcrrc_id'].notna())
].copy()

print(f"df_raw shape after exclusion: {df_raw.shape}")
print(f"Outcome distribution (child_face_position_fi___1):")
print(df_raw['child_face_position_fi___1'].value_counts(dropna=False))

Mounted at /content/drive
df_raw shape after exclusion: (317, 500)
Outcome distribution (child_face_position_fi___1):
child_face_position_fi___1
1.0    175
0.0    142
Name: count, dtype: int64


## Cell 2 — Column Dropping

Four sequential stages. Each produces a named copy so earlier frames stay
accessible. Warnings are printed (not errors) when a column is already gone
from an earlier stage.

**Order:**
1. Low-variance (dynamic, regex on one-hot columns)
2. Red columns (Laura's explicit list)
3. Sentinel columns (dynamic regex on column names)
4. Explicit lists (zero-variance, collinear, high-missingness, other)

In [ ]:
# ── 2a. LOW-VARIANCE DROP ─────────────────────────────────────────────────────
# One-hot columns (name contains '___') where positive count (sum) <= threshold.
# Runs first so later explicit lists don't need to enumerate every sparse one-hot.

LOW_VAR_THRESHOLD = 2

oh_cols = [c for c in df_raw.columns if '___' in c]
LOW_VARIANCE_COLS = [
    c for c in oh_cols
    if pd.to_numeric(df_raw[c], errors='coerce').sum() <= LOW_VAR_THRESHOLD
]

df_low_var_dropped = df_raw.drop(columns=LOW_VARIANCE_COLS).copy()

print(f"[2a] Low-variance drop  (threshold: sum <= {LOW_VAR_THRESHOLD})")
print(f"     Dropped : {len(LOW_VARIANCE_COLS)}  |  Remaining : {df_low_var_dropped.shape[1]}")

[2a] Low-variance drop  (threshold: sum <= 2)
     Dropped : 121  |  Remaining : 379


In [ ]:
# ── 2b. RED COLUMNS DROP ──────────────────────────────────────────────────────
# RED columns are marked by clinicians as non-informative. Dropped.

RED_COLS = [
    # free text / metadata / administrative
    'notes', 'fi_not_completed', 'clinical_medical_records_review_complete',
    'opine_mod_orig_mec', 'Unnamed: 464', 'c', 'opine_fcod_orig_path',

    # pregnancy / birth / newborn screening
    'pregnancy_pmh_details',
    'newborn_hear_test_performed',   # performed flag; content parsed to binary separately
    'newborn_hear_test_results',     # free-text; replaced by newborn_hear_test_results_all_normal
    'newborn_screen_test_performed', # performed flag; content parsed to binary separately
    'newborn_screen_test_results',   # free-text; replaced by newborn_screen_test_results_all_normal
    'sudc_childs_delivery___998',

    # neurological condition one-hots (index child)
    'neuro_condit___2', 'neuro_condit___3',
    'neuro_condit___7', 'neuro_condit___8', 'neuro_condit___9',
    'neuro_condit___10', 'neuro_condit___998', 'neuro_condit___99',

    # febrile seizure / epilepsy / autism / ID / neuro-other — index child flags
    # kept: febrile_sz___any_type_Final (summary variable), family-history suffixes 2-7
    'febrile_sz___1', 'f+EA:KHebrile_sz___8',  # garbled column name; both refer to index child
    'epilepsy___1', 'epilepsy___8',
    'autism___1', 'autism___8',
    'intellectual_disabilities___1', 'intellectual_disabilities___8',
    'neuro_other___1', 'neuro_other___8',

    # seizure categorisation sentinels
    'seizures_were_categorized___3', 'seizures_were_categorized___4',
    'seizures_were_categorized___5',

    # cardiac condition sentinels / administrative
    'cardiaccondit___999', 'cardiaccondit___998', 'cardiaccondit___99',

    # cardiac family-history index-child flags (index = ___1 / ___8)
    'cardiac_arrhythmia___1', 'cardiac_arrhythmia___8',
    'sudden_cardiac_death___1', 'sudden_cardiac_death___8',
    'syncope___1',

    # cardiac other (all sub-categories)
    'cardiac_other___1', 'cardiac_other___2', 'cardiac_other___3',
    'cardiac_other___4', 'cardiac_other___5', 'cardiac_other___6',
    'cardiac_other___7', 'cardiac_other___8',

    # respiratory sentinels / duplicates
    'respcondit___3', 'respcondit___6', 'respcondit___998', 'respcondit___99',

    # asthma / sleep apnea index-child flags
    'asthma___1', 'asthma___8',
    'sleep_apnea___1', 'sleep_apnea___8',

    # psychiatric sentinels
    'psychcondit___2', 'psychcondit___99', 'psychcondit___998',

    # metabolic disease (all sub-categories)
    'metabolic_disease___1', 'metabolic_disease___2', 'metabolic_disease___3',
    'metabolic_disease___4', 'metabolic_disease___5', 'metabolic_disease___6',
    'metabolic_disease___7', 'metabolic_disease___8',

    # genetic history sentinels
    'genetic_hx___3', 'genetic_hx___99',

    # family / medication administrative
    'vaccine_tolerance_fam', 'fertility_meds___99', 'term_meds_fa',

    # recent activity sentinels / unknowns
    'recent_actvity_hx___8', 'recent_actvity_hx___9',

    # last 48h symptoms — None / I-don't-recall / other sentinels
    'last_48hr_sxs___1', 'last_48hr_sxs___8', 'last_48hr_sxs___10',

    # exercise history (all sub-categories)
    'sudc_exer_hx___0', 'sudc_exer_hx___1', 'sudc_exer_hx___2',
    'sudc_exer_hx___3', 'sudc_exer_hx___4', 'sudc_exer_hx___5',
    'sudc_exer_hx___6', 'sudc_exer_hx___7', 'sudc_exer_hx___8',
    'sudc_exer_hx___998', 'sudc_exer_hx___99', 'sudc_exer_hx___999',

    # terminal triggers (all sub-categories)
    'term_triggers___1', 'term_triggers___2', 'term_triggers___3',
    'term_triggers___4', 'term_triggers___5', 'term_triggers___6',
    'term_triggers___7', 'term_triggers___8', 'term_triggers___9',
    'term_triggers___99', 'term_triggers___998',

    # death scene face position (all except ___1 which is the outcome)
    'child_face_position_fi___2', 'child_face_position_fi___3',
    'child_face_position_fi___4', 'child_face_position_fi___5',
    'child_face_position_fi___6', 'child_face_position_fi___7',

    # cause of death / classification fields
    'sudden_explained_death___1', 'sudden_explained_death___8',
    'sudden_unexplained_death___1', 'sudden_unexplained_death___8',
    'fpc_dc_1bcd', 'fpc_dc_line2', 'dc_line_1a_fpc',
    'intrinsic_factors_usd', 'extrinsic_factors_usd', 'original_vs_fpc_cod',

    # sleep location (removed per Laura; to be re-added via sleep_environ if available)
    'start_sleep_location_fi',

    # anthropometric — not in scope for this model
    'bmi', 'head_circumference', 'weight_length',

    # free-text sleep description
    'desc_favesleep',
]

already_gone_red = [c for c in RED_COLS if c not in df_low_var_dropped.columns]
still_present_red = [c for c in RED_COLS if c in df_low_var_dropped.columns]

for c in already_gone_red:
    print(f"  [WARNING] RED_COLS: '{c}' not found — already removed by low-variance filter")

df_red_dropped = df_low_var_dropped.drop(columns=still_present_red).copy()

print(f"\n[2b] Red columns drop")
print(f"     Entries in RED_COLS      : {len(RED_COLS)}")
print(f"     Already gone (low-var)   : {len(already_gone_red)}")
print(f"     Actually dropped now     : {len(still_present_red)}")
print(f"     Remaining cols           : {df_red_dropped.shape[1]}")

  [WARNING] RED_COLS: 'neuro_condit___3' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'neuro_condit___7' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'neuro_condit___8' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'neuro_condit___9' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'neuro_condit___10' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'epilepsy___1' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'autism___1' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'seizures_were_categorized___3' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'cardiaccondit___999' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'cardiaccondit___99' not found — already removed by low-variance filter
  [WARNING] RED_COLS: 'sudden_cardiac_death___1' not found — already re

In [ ]:
# ── 2c. SENTINEL COLUMNS DROP ─────────────────────────────────────────────────
# Columns whose names end in 99, 998, 999, or _0 represent
# "unknown / N/A / none-of-the-above" sentinel indicators.
# Identified dynamically by regex so new sentinel columns are caught automatically.

SENTINEL_REGEX = r'99$|998$|999$|_0$'

SENTINEL_COLS = [
    c for c in df_red_dropped.columns
    if re.search(SENTINEL_REGEX, c)
]

df_sentinel_dropped = df_red_dropped.drop(columns=SENTINEL_COLS).copy()

print(f"[2c] Sentinel columns drop  (regex: '{SENTINEL_REGEX}')")
print(f"     Dropped : {len(SENTINEL_COLS)}  |  Remaining : {df_sentinel_dropped.shape[1]}")
print(f"     Cols dropped: {SENTINEL_COLS}")

[2c] Sentinel columns drop  (regex: '99$|998$|999$|_0$')
     Dropped : 8  |  Remaining : 283
     Cols dropped: ['pmh_pregnancy___998', 'pmh_pregnancy___99', 'in_hospital_newborn_hx___998', 'rheu_inflam___0', 'rheu_inflam___998', 'genetic_hx___998', 'sudc_health_hx___998', 'sudc_health_hx___0']


In [ ]:
# ── 2d. EXPLICIT LIST DROPS ───────────────────────────────────────────────────
# Zero-variance, collinear, high-missingness, and other drops combined
# into one pass (all lists are fully explicit and don't interact).

ZERO_VAR_COLS = [
    # Confirmed zero-variance (all same value) across post-exclusion sample
    'pmh_pregnancy___1', 'pmh_pregnancy___5', 'pmh_pregnancy___12',
    'pmh_pregnancy___37', 'in_hospital_newborn_hx___19',
    'in_hospital_newborn_hx___20', 'rheu_inflam___3', 'rheu_inflam___4',
    'rheu_inflam___5', 'rheu_inflam___6', 'rheu_inflam___7',
    'rheu_inflam___8', 'rheu_inflam___9', 'rheu_inflam___10',
    'cardiaccondit___3', 'cardiaccondit___4', 'cardiaccondit___5',
    'cardiaccondit___6', 'psychcondit___3', 'genetic_hx___2',
    'sudc_health_hx___2', 'sudc_health_hx___14', 'autism___2', 'autism___3',
    'sudden_cardiac_death___2', 'sudden_cardiac_death___3',
    'sudden_cardiac_death___5', 'syncope___4', 'syncope___5', 'syncope___8',
    'sudden_explained_death___5', 'sudden_unexplained_death___2',
    'sudden_unexplained_death___3', 'sudden_unexplained_death___5',
    'bedsharing_details_fi___7', 'bedsharing_details_fi___8',
    'bedsharing_details_fi___9',
]

COLLINEAR_COLS = [
    # corr >= 0.978 with a retained variable
    'labor_details___8',            # corr=0.986 with sudc_childs_delivery___1/2; delivery type duplication
    'psychcondit___5',              # corr=1.000 with seizure
    'sudc_health_hx___1',           # corr=1.000 with seizure; redundant encoding
    'tachypnea',                    # corr=1.000 with tachycardia; keeping tachycardia
    'difficulty_regulation_temp',   # corr=0.999 with require_cpr
    'vent',                         # corr=1.000 with require_cpr; redundant CPR indicator
]

HIGH_MISSINGNESS_COLS = [
    # Missingness > 35% in post-exclusion sample
    'sleephrspernight',             # 40.7% missing
    'snoring',                      # 86.4% missing; mode imputation would fabricate signal
                                     # (observed prevalence 18.6% -> imputed prevalence 2.5%)
]

OTHER_DROP_COLS = [
    # Redundant body position encoding — body_position_prone re-engineered from df_raw in step 3
    'child_body_position_fi___2',

    # Parasomnia sentinels / unknowns
    'parasomnias___8', 'parasomnias___9',

    # Bedsharing NA sentinel
    # NOTE: ___3 (sharing with >10yr child, n=2) was removed by low-variance filter.
    # Intentionally not rescued — n=2 too sparse to contribute meaningful signal.
    # ___6 (n=1) and ___10 (n=1) similarly excluded.
    'bedsharing_details_fi___11',

    # Symptom sentinel
    'last_48hr_sxs___14',

    # Family history: full sibling #1 removed (keeping 2nd-degree / parent level for consistency)
    'sudden_unexplained_death___4', 'sudden_explained_death___4',
    'sudden_cardiac_death___4',

    # Obstetric sentinels not caught by name-based regex
    'pmh_pregnancy___998', 'pmh_pregnancy___41', 'pmh_pregnancy___99',
    'labor_details___12', 'labor_details___13',
    'in_hospital_newborn_hx___25', 'in_hospital_newborn_hx___26',
    'in_hospital_newborn_hx___998', 'in_hospital_newborn_hx___999',
    'rheu_inflam___0', 'rheu_inflam___998', 'rheu_inflam___999',
    'neuro_condit___11', 'neuro_condit___13',
    'genetic_hx___998', 'fertility_meds___4',
    'sudc_health_hx___0', 'sudc_health_hx___99', 'sudc_health_hx___998',

    # Death scene
    'found_location_fi',

    # Pregnancy variables: gestational Rx (other) / cardiovascular other — too sparse
    'pmh_pregnancy___29', 'cardiaccondit___10',

    # Delivery: vaginal/cesarean raw flags replaced by engineered vaginal_birth in step 3
    'sudc_childs_delivery___1', 'sudc_childs_delivery___2',

    # Symptom: injury/other — too heterogeneous
    'last_48hr_sxs___13',

    # Full-sibling autism / arrhythmia (sibling #2; sibling #1 already zero-var)
    'autism___4', 'cardiac_arrhythmia___4',

    # Cigarettes: redundant with cigarette_smoking_during_pregnancy (medical records = higher quality)
    'cigarettes',

    # Dataset inclusion criterion — not a predictor
    'death_sleep_period_fi',

    # Dropped per Laura's review
    'parasomnias___1',

    # Developmental: keeping only development_wnl (broadest, medical-record sourced)
    # development_wnl (n=44 abnormal) subsumes neuro_condit___6 (n=26) and sudc_health_hx___6 (n=19)
    'neuro_condit___6',
    'sudc_health_hx___6',

    # Immunoglobulins abnormal: n=3 positive, near-zero variance; Laura unclear on definition
    'rheu_inflam___11',

    # Birth order / multiples: preg_multiple dropped (birth_order_multiple already dropped)
    'preg_multiple',

    # ER visits: keeping number_er_visits (actual count); dropping ordinal Likert scale
    'er_visits',

    # Hospitalizations: keeping number_of_hospital_adm (actual count); dropping ordinal scale
    'hospitalizations',

    # Residence type: not mechanistically linked to prone sleep death
    'type_residence',

    # Vaccine timing: only n=1 case within ≤2 weeks of death — not informative as binary;
    # NOTE: revisit if cohort expands
    'last_vacc_to_death',

    # redundant information with birthlength_percent, birthweight_percent
    'bl_percentile', 'bw_percentile',

    # redundant information with 'pmh_pregnancy___' 15, 18, 28, which came from medical records
    'otc_prenatal_vitamins', 'otc_acetominophen', 'cannabi',

    # redundant information with 'in_hospital_newborn_hx___' 10, 2, 23, 16
    'tachycardia', 'oxygen', 'mom_child_dc', 'respiratory_distress',

    # redundant information with 'sudc_health_hx___4'
    'turn_blue',

    # redundant information with 'respcondit___1'
    'sudc_health_hx___19',

    # redundant information with 'development_wnl'
    'milestones',

    # redundant information with 'respcondit___1'
    'sudc_health_hx___19',

]

EXPLICIT_DROP_COLS = list(dict.fromkeys(
    ZERO_VAR_COLS + COLLINEAR_COLS + HIGH_MISSINGNESS_COLS + OTHER_DROP_COLS
))

not_found = [c for c in EXPLICIT_DROP_COLS if c not in df_sentinel_dropped.columns]
found     = [c for c in EXPLICIT_DROP_COLS if c in df_sentinel_dropped.columns]

for c in not_found:
    print(f"  [WARNING] EXPLICIT_DROP: '{c}' not found — already removed by earlier stage")

df_dropped = df_sentinel_dropped.drop(columns=found).copy()

print(f"\n[2d] Explicit list drop")
print(f"     Total entries            : {len(EXPLICIT_DROP_COLS)}")
print(f"     Already gone             : {len(not_found)}")
print(f"     Actually dropped now     : {len(found)}")
print(f"     Remaining cols           : {df_dropped.shape[1]}")
print(f"\nFinal df_dropped shape: {df_dropped.shape}")

  [WARNING] EXPLICIT_DROP: 'pmh_pregnancy___1' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'pmh_pregnancy___5' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'pmh_pregnancy___12' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'pmh_pregnancy___37' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'in_hospital_newborn_hx___19' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'in_hospital_newborn_hx___20' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'rheu_inflam___3' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'rheu_inflam___4' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'rheu_inflam___5' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'rheu_inflam___6' not found — already removed by earlier stage
  [WARNING] EXPLICIT_DROP: 'rheu_inflam___7' not found — already r

## Cell 3 — Helper Functions for Recoding

In [ ]:
def parse_hearing(val):
    """
    newborn_hear_test_results free text -> binary.
    1 = passed (bilaterally or ENT-confirmed after initial failure)
    0 = any failure documented (initial, unilateral, or unrepeated referral)
    NaN = no records, unavailable, or uninterpretable entry

    Final distribution: 1=251, 0=13, NaN=53
    """
    if pd.isna(val): return np.nan
    v_orig = val.strip()
    v = v_orig.lower()

    # --- Explicit overrides (evaluated before general patterns) ---
    # ENT-confirmed bilateral pass after initial failure -> 1
    if 'ENT' in v_orig and 'passed' in v: return 1.0
    # Misrouted screen test entry in hearing field -> NaN
    if 'hypothyroidism' in v and 'hearing' not in v and 'pass' not in v: return np.nan
    # OAE performed but no pass/fail result recorded -> NaN
    if 'oae test done bilaterally' in v and 'pass' not in v and 'fail' not in v: return np.nan

    # --- No record -> NaN (only if no pass/normal also mentioned) ---
    no_record = ['no report', 'no records', 'not available', 'unavailable', 'unk', 'unknown']
    if any(p in v for p in no_record) and 'pass' not in v and 'normal' not in v: return np.nan

    # --- Any failure -> 0 ---
    if any(p in v for p in ['fail', 'failed', 'refer', 'did not pass']): return 0.0

    # --- Pass -> 1 ---
    if any(p in v for p in ['pass', 'passed', 'normal']): return 1.0

    return np.nan


def parse_screen(val):
    """
    newborn_screen_test_results free text -> binary.
    1 = all results normal/negative (including transiently abnormal but confirmed resolved)
    0 = any confirmed abnormal finding, including genetic traits (e.g. sickle cell trait)
    NaN = no records or results unavailable

    Final distribution: 1=229, 0=7, NaN=81

    Key edge cases resolved:
    - 'No abnormalities' -> 1  ('abnormal' is a false-positive substring)
    - 'sickle cell trait' -> 0  (tightened to avoid 'sickle cell screening not suspected')
    - CAH elevated but 2nd-tier test negative -> 1
    - Initial abnormal amino acid, repeat normal -> 1
    - Transient T4/TSH, explicitly resolved -> 1
    """
    if pd.isna(val): return np.nan
    v_orig = val.strip()
    v = v_orig.lower()

    # --- Explicit overrides ---
    if v.startswith('no abnormalit'): return 1.0
    if ('2nd tier' in v or 'second tier' in v) and 'negative' in v: return 1.0
    if 'initial abnormal' in v and 'repeat' in v and 'normal' in v: return 1.0
    if 'resolved' in v and ('t4' in v or 'tsh' in v or 'below normal' in v): return 1.0
    if 'sickle cell trait' in v: return 0.0

    # --- No record -> NaN, unless secondary source confirms normal ---
    no_record = [
        'no report', 'no records', 'not available', 'unavailable', 'unknown', 'unk',
        'results not available', 'no reported results', 'mentioned to be performed',
        'testing mentioned', 'pediatric records indicate', 'screen was mentioned',
        'specimen received'
    ]
    if any(p in v for p in no_record):
        if 'normal' in v or 'negative' in v: return 1.0
        return np.nan

    # --- Abnormal -> 0 ---
    if any(p in v for p in ['abnormal', 'elevated', 'above normal', 'below normal',
                             'g6pd:+', 'suggestive of', 'slightly elevated', 'concern for']):
        return 0.0

    # --- Normal -> 1 ---
    if any(p in v for p in ['normal', 'negative', 'wnl', 'within normal', 'within acceptable',
                             'all in range', 'all negative', 'all normal', 'pass', 'passed',
                             'everything was normal', 'everything was negative',
                             'everything within normal']):
        return 1.0

    return np.nan


def parse_percentile(val):
    """
    Parse messy string percentile fields (bw_percentile, bl_percentile) -> float.

    Handles:
    - Plain numbers: '75' -> 75.0
    - Ordinal suffixes: '75th' -> 75.0
    - Ranges: '50-75' -> 62.5 (midpoint)
    - Bounds: '>95' -> 95.0, '<3' -> 3.0
    - Unknown/not described -> NaN
    - Excel date artifacts ('10-Mar', '25-Oct') -> NaN (month name present)
    - Values outside [0, 100] -> NaN
    """
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()

    if any(x in s for x in ['unknown', 'not described', 'not provided', 'unk']): return np.nan

    # Excel date artifacts (e.g. '10-Mar', '25-Oct') — month name present
    months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
    if any(m in s for m in months): return np.nan

    # Strip ordinal suffixes and prose
    s = re.sub(r'(st|nd|rd|th)\b', '', s)
    s = re.sub(r'(percentile|per pm report|for age|cdc|weight vs age.*|length vs age.*)', '', s)
    s = s.strip().rstrip('.')

    # Greater-than / less-than: use boundary value
    gt = re.match(r'^[>≥]\s*(\d+\.?\d*)', s)
    lt = re.match(r'^[<≤]\s*(\d+\.?\d*)', s)
    if gt: return float(gt.group(1))
    if lt: return float(lt.group(1))

    # Range 'X-Y' -> midpoint
    rng = re.match(r'^(\d+\.?\d*)\s*[-–to]+(\d+\.?\d*)', s)
    if rng: return (float(rng.group(1)) + float(rng.group(2))) / 2

    # Plain number
    num = re.match(r'^(\d+\.?\d*)$', s.strip())
    if num:
        v = float(num.group(1))
        return v if 0 <= v <= 100 else np.nan

    return np.nan


def parse_sick_visits(val):
    """
    Parse sick_visits_per_year string -> float.
    '<1' -> 0.5, 'unk'/'unknown' -> NaN, plain floats parsed directly.
    """
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    if any(x in s for x in ['unk', 'unknown', 'missing']): return np.nan
    if s.startswith('<'): return 0.5
    try: return float(s)
    except: return np.nan


def collapse_family_history(df, prefix, suffix_range, new_col):
    """
    OR-collapse a set of family-history one-hot columns into a single binary.
    Any suffix in suffix_range not present in df is silently skipped.
    Source columns are dropped after collapse.
    """
    cols = [f"{prefix}___{i}" for i in suffix_range]
    existing = [c for c in cols if c in df.columns]
    if not existing:
        print(f"  [WARNING] collapse_family_history: no columns found for prefix '{prefix}'")
        return df
    df[new_col] = (df[existing] == 1).any(axis=1).astype(int)
    df.drop(columns=existing, inplace=True)
    return df


print("Helper functions defined.")

Helper functions defined.


## Cell 4 — Recoding

`recode_sudc_data(df_dropped, df_raw)` takes both the dropped frame and `df_raw`.
Columns removed by the drop pipeline but needed for feature engineering are
pulled from `df_raw` directly (e.g. ethnicity, vaginal_birth, born_full_term,
body_position_prone, newborn test texts).

Sub-sections:
- **3a** Text parsing (newborn test results)
- **3b** Binary mappings
- **3c** Collapse / engineer new columns
- **3d** Sentinel replacement on count/continuous variables
- **3e** String column parsing (percentiles, sick visits)
- **3f** Family history aggregations
- **3g** Remaining one-hot cleanup

In [ ]:
def recode_sudc_data(df_dropped, df_raw):
    """
    Recode all variables. df_raw is passed so columns removed by the drop
    pipeline can still be referenced when building engineered features.
    Returns df_recoded with the same row count as df_dropped.
    """
    df = df_dropped.copy()

    # ── 3a. TEXT PARSING ──────────────────────────────────────────────────────
    # Source columns were removed in RED_COLS; pull from df_raw.
    df['newborn_hear_test_results_all_normal'] =         df_raw['newborn_hear_test_results'].apply(parse_hearing)
    df['newborn_screen_test_results_all_normal'] =       df_raw['newborn_screen_test_results'].apply(parse_screen)

    # ── 3b. BINARY MAPPINGS ───────────────────────────────────────────────────

    # Standard binary: 1=yes, 2=no, 99=unknown -> NaN
    std_binary = {1: 1, 2: 0, 99: np.nan}
    std_binary_cols = [
        'rolled', 'crawled', 'started', 'first_words', 'vaccines_uptodate',
        'development_wnl', 'preg_complications',
        'delivery_complications', 'special_needs',
    ]
    for col in [c for c in std_binary_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(std_binary)

    # Neonatal complications: 1=yes, 2=no, 99=unknown -> NaN
    neonatal_cols = [
        'require_cpr', 'stop_breathing',
        'bradycardia', 'seizure',
    ]
    for col in [c for c in neonatal_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').map({1: 1, 2: 0, 99: np.nan})

    # Milestone binary: 1=ahead/2=on-track -> 0 (no concern),
    # 3=possibly delayed -> 1 (concern), 4=unknown -> NaN
    # NOTE: 3='possibly' is conservatively mapped to 1 per Laura's direction
    milestone_cols = ['milestones', 'expressive_speech', 'comprehension',
                      'fine_motor', 'gross_motor', 'behavior']
    for col in [c for c in milestone_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').map({1: 0, 2: 0, 3: 1, 4: np.nan})

    # Sleep quality binary: 1=yes/problem, 2=no, 3=I don't know -> NaN
    sleep_binary_cols = ['snoring', 'fallingasleep', 'stayingasleep', 'favesleep']
    for col in [c for c in sleep_binary_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').map({1: 1, 2: 0, 3: np.nan})

    # awokeoften: 1=never, 2=occasionally -> 0 (rarely/never)
    #             3=sometimes, 4=always   -> 1 (daily/weekly)
    #             5=N/A                   -> NaN
    if 'awokeoften' in df.columns:
        df['awokeoften'] = pd.to_numeric(df['awokeoften'], errors='coerce').map(
            {1: 0, 2: 0, 3: 1, 4: 1, 5: np.nan})

    # Substance binary: 1=none -> 0, 2-5=any use -> 1, 99=unknown -> NaN
    substance_cols = [
        'alcohol', 'caffeine_coffee_tea', 'cocaine', 'amphetamines', 'opiods_heroin'
    ]
    for col in [c for c in substance_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(
            {1: 0, 2: 1, 3: 1, 4: 1, 5: 1, 99: np.nan})

    # allergy_hx: 1=yes, 2=no, 3=possibly -> 1 (conservative), 99=NaN
    # NOTE: 3='possibly allergic' treated as yes per clinical conservatism
    if 'allergy_hx' in df.columns:
        df['allergy_hx'] = pd.to_numeric(df['allergy_hx'], errors='coerce').map(
            {1: 1, 2: 0, 3: 1, 99: np.nan})

    # gender -> female binary (1=male -> 0, 2=female -> 1)
    if 'gender' in df.columns:
        df['female'] = pd.to_numeric(df['gender'], errors='coerce').map({1: 0, 2: 1})
        df.drop(columns='gender', inplace=True)

    # breastfed -> breastfed_3m_plus (1=never, 2=<1m -> 0; 3=1-3m, 4=3-6m, 5=>6m -> 1)
    if 'breastfed' in df.columns:
        df['breastfed_3m_plus'] = pd.to_numeric(df['breastfed'], errors='coerce').map(
            {1: 0, 2: 0, 3: 1, 4: 1, 5: 1, 99: np.nan})
        df.drop(columns='breastfed', inplace=True)

    # mother_marital_status -> mother_married (1=married -> 1, all else -> 0, 998/99 -> NaN)
    if 'mother_marital_status' in df.columns:
        df['mother_married'] = pd.to_numeric(
            df['mother_marital_status'], errors='coerce').apply(
            lambda x: 1.0 if x == 1 else (np.nan if x in [998, 99] else 0.0))
        df.drop(columns='mother_marital_status', inplace=True)

    # resident_smokers -> any_resident_smokers (0=none -> 0, 1/2/3=any -> 1)
    if 'resident_smokers' in df.columns:
        df['any_resident_smokers'] = pd.to_numeric(
            df['resident_smokers'], errors='coerce').apply(
            lambda x: 0.0 if x == 0 else (1.0 if x in [1, 2, 3] else np.nan))
        df.drop(columns='resident_smokers', inplace=True)

    # mother_premie_hx -> any_mother_premie
    # Encodes: of mother's live births, how many were premature (<37 weeks)?
    # Collapsed to binary: 0=none, 1=any, 99=NaN
    if 'mother_premie_hx' in df.columns:
        df['any_mother_premie'] = pd.to_numeric(
            df['mother_premie_hx'], errors='coerce').apply(
            lambda x: np.nan if x == 99 else (0.0 if x == 0 else 1.0))
        df.drop(columns='mother_premie_hx', inplace=True)

    # prenatal_care_start -> prenatal_care_start_before_2nd_trimester
    # 1=1st tri -> 1, 2=2nd tri/3=3rd tri/4=no care -> 0, 5=unknown -> NaN
    if 'prenatal_care_start' in df.columns:
        df['prenatal_care_start_before_2nd_trimester'] = pd.to_numeric(
            df['prenatal_care_start'], errors='coerce').map(
            {1: 1, 2: 0, 3: 0, 4: 0, 5: np.nan})
        df.drop(columns='prenatal_care_start', inplace=True)

    # ── 3c. COLLAPSE / ENGINEER NEW COLUMNS ───────────────────────────────────

    # vaginal_birth: from df_raw, using sudc_childs_delivery___1/___2 (both cols were
    # in OTHER_DROP_COLS, dropped on the assumption they'd be redundant with this
    # engineered feature).
    #
    # sudc_childs_delivery___1 (vaginal) / ___2 (cesarean) has only
    # 1/317 missing (99.7% coverage) vs. the previously-used labor_details___7/___10
    # pairing, which had 121/317 missing (61.8% coverage). Root cause: labor_details___10
    # is NOT a general cesarean flag -- it correlates near-zero with sudc_childs_delivery___2
    # (true cesarean). The 3 cases where labor_details___10=1 but sudc_childs_delivery___1
    # (vaginal)=1 (subjects 106, 336, 372) all co-occur with forceps/vacuum flags, indicating
    # labor_details___10 actually captures assisted/operative VAGINAL delivery, not cesarean.
    # labor_details___8 (dropped as COLLINEAR_COLS) is the true cesarean-concordant column
    # (r=0.978 with sudc_childs_delivery___2), which confirms sudc_childs_delivery is the
    # more reliable source field for delivery mode overall.
    # Where both known (n=196), labor_details/sudc_childs_delivery agree 98.5% of the time.
    if {'sudc_childs_delivery___1', 'sudc_childs_delivery___2'}.issubset(df_raw.columns):
        vag = pd.to_numeric(df_raw['sudc_childs_delivery___1'], errors='coerce') == 1
        ces = pd.to_numeric(df_raw['sudc_childs_delivery___2'], errors='coerce') == 1
        df['vaginal_birth'] = np.where(vag, 1.0, np.where(ces, 0.0, np.nan))

    # Drop raw source columns now that vaginal_birth has been engineered.
    # labor_details___7/___10 are also dropped here: they're the inferior/superseded
    # source for the same clinical concept and were never independently named in
    # RENAME_MAP, so keeping them as separate raw features would just duplicate
    # vaginal_birth with more missingness and one mismatched category.
    for col in ['labor_details___7', 'labor_details___10',
                'sudc_childs_delivery___1', 'sudc_childs_delivery___2']:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

    # delivery_require_forceps / delivery_require_vacuum
    # sudc_childs_delivery___3 and ___4 survive the drop pipeline
    if 'sudc_childs_delivery___3' in df.columns:
        df['delivery_require_forceps'] = pd.to_numeric(
            df['sudc_childs_delivery___3'], errors='coerce').map({1: 1, 0: 0})
        df.drop(columns='sudc_childs_delivery___3', inplace=True)

    if 'sudc_childs_delivery___4' in df.columns:
        df['delivery_require_vacuum'] = pd.to_numeric(
            df['sudc_childs_delivery___4'], errors='coerce').map({1: 1, 0: 0})
        df.drop(columns='sudc_childs_delivery___4', inplace=True)

    # born_full_term: from df_raw to avoid NaN->0 coercion bug
    # gestation: 1=full term, 2=late preterm, 3=preterm, 4=very preterm, 99=unknown
    # IMPORTANT: must use df_raw here; using df after sentinel replacement would
    # coerce gestation=99 -> NaN -> 0 via astype(int), incorrectly labeling as not full term
    if 'gestation' in df_raw.columns:
        df['born_full_term'] = pd.to_numeric(df_raw['gestation'], errors='coerce').map(
            {1: 1, 2: 0, 3: 0, 4: 0, 99: np.nan})
    if 'gestation' in df.columns:
        df.drop(columns='gestation', inplace=True)

    # conception_duration -> conception_duration_over_12m
    # 100=<3m, 101=3-12m -> 0 (not over 12m)
    # 102=12-18m, 103=18-24m, 104=24m+ -> 1 (over 12m)
    if 'conception_duration' in df.columns:
        df['conception_duration_over_12m'] = pd.to_numeric(
            df['conception_duration'], errors='coerce').map(
            {100: 0, 101: 0, 102: 1, 103: 1, 104: 1})
        df.drop(columns='conception_duration', inplace=True)

    # any_bed_sharing: collapse bedsharing_details_fi___1, ___2, ___4, ___5
    # NOTE: ___3 (sharing with >10yr child, n=2) removed by low-variance filter.
    # Not rescued from df_raw — n=2 is too sparse to contribute meaningful signal.
    # ___6 (n=1) and ___10 (n=1) also excluded for the same reason.
    bed_cols = [f'bedsharing_details_fi___{i}' for i in [1, 2, 4, 5]]
    existing_bed = [c for c in bed_cols if c in df.columns]
    if existing_bed:
        df['any_bed_sharing'] = (
            df[existing_bed].apply(pd.to_numeric, errors='coerce') == 1
        ).any(axis=1).astype(int)
        df.drop(columns=existing_bed, inplace=True)

    # body_position_prone/supine/side: mutually-exclusive-ish position flags.
    # 8 subjects have raw position = "other" (___5) or "don't know"
    # (___6), or all six raw flags blank/zero. Previously these silently fell out as
    # 0/0/0 across all three engineered columns -- indistinguishable from a confirmed
    # "not in any of these positions". We now mark them NaN so the existing binary
    # mode-imputation step (4c) handles them consistently with every other feature,
    # instead of hard-coding a false negative on the paper's central predictor.
    _pos_raw_cols = [f'child_body_position_fi___{i}' for i in [1, 2, 3, 4, 5, 6]]
    if set(_pos_raw_cols).issubset(df_raw.columns):
        _pos_raw = df_raw[_pos_raw_cols].apply(pd.to_numeric, errors='coerce')
        # "Don't know" (___6) / "other" (___5) are secondary to a specific position flag.
        # Only treat as genuinely unknown when NONE of the specific position flags
        # (___1 supine, ___2 prone, ___3/___4 side) are set -- i.e. don't-know/other
        # is the *only* signal present, not merely co-checked alongside a real answer.
        _specific_pos_cols = ['child_body_position_fi___1', 'child_body_position_fi___2',
                               'child_body_position_fi___3', 'child_body_position_fi___4']
        UNKNOWN_POSITION_MASK = (_pos_raw[_specific_pos_cols].fillna(0).sum(axis=1) == 0)
    else:
        UNKNOWN_POSITION_MASK = pd.Series(False, index=df.index)

    # body_position_prone: from df_raw (child_body_position_fi___2 was in OTHER_DROP_COLS)
    if 'child_body_position_fi___2' in df_raw.columns:
        df['body_position_prone'] = pd.to_numeric(
            df_raw['child_body_position_fi___2'], errors='coerce').map({1: 1, 0: 0})

    # body_position_supine: ___1 survives drop pipeline
    if 'child_body_position_fi___1' in df.columns:
        df['body_position_supine'] = pd.to_numeric(
            df['child_body_position_fi___1'], errors='coerce').map({1: 1, 0: 0})
        df.drop(columns='child_body_position_fi___1', inplace=True)

    # body_position_side: collapse ___3 (right) and ___4 (left)
    # NOTE: ___8 (side-unspecified) had sum=0, correctly removed by low-variance filter
    side_cols = [c for c in ['child_body_position_fi___3', 'child_body_position_fi___4']
                 if c in df.columns]
    if side_cols:
        df['body_position_side'] = (
            df[side_cols].apply(pd.to_numeric, errors='coerce') == 1
        ).any(axis=1).astype(int)
        df.drop(columns=side_cols, inplace=True)

    # Apply the unknown-position mask (see note above). Uses .values-aligned boolean
    # array since UNKNOWN_POSITION_MASK was built from df_raw (same row order/index as df).
    for col in ['body_position_prone', 'body_position_supine', 'body_position_side']:
        if col in df.columns:
            df.loc[UNKNOWN_POSITION_MASK, col] = np.nan

    # Drop remaining body position cols (Other/Don't Know — too sparse/uninformative)
    for col in ['child_body_position_fi___5', 'child_body_position_fi___6']:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

    # Ethnicity collapse: pulled from df_raw because ___1 (African, n=1) and ___3 (Asian, n=1)
    # were removed by the low-variance filter, but must be included to correctly
    # identify all Black and Asian subjects
    df['ethnicity_white']    = (df_raw['sudc_child_ethnicity___8'].fillna(0) == 1).astype(int)
    df['ethnicity_black']    = (
        (df_raw['sudc_child_ethnicity___1'].fillna(0) == 1) |
        (df_raw['sudc_child_ethnicity___2'].fillna(0) == 1)
    ).astype(int)
    df['ethnicity_hispanic'] = (df_raw['sudc_child_ethnicity___6'].fillna(0) == 1).astype(int)
    df['ethnicity_asian']    = (
        (df_raw['sudc_child_ethnicity___3'].fillna(0) == 1) |
        (df_raw['sudc_child_ethnicity___4'].fillna(0) == 1) |
        (df_raw['sudc_child_ethnicity___5'].fillna(0) == 1)
    ).astype(int)
    # Drop any ethnicity raw cols that survived the drop pipeline
    eth_cols_to_drop = [c for c in df.columns if 'sudc_child_ethnicity' in c]
    if eth_cols_to_drop:
        df.drop(columns=eth_cols_to_drop, inplace=True)

    # blood_mismatch: DROPPED (53.3% missing -- consistent with the same
    # missingness-driven exclusion principle used for special_needs/sleephrspernight/
    # snoring). Source columns dropped directly instead of engineering the feature.
    for col in ['baby_blood_type', 'maternal_blood_type']:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

    # ── 3d. SENTINEL REPLACEMENT ON COUNT / CONTINUOUS VARIABLES ─────────────
    # Must run before binary collapse of abortions / miscarriage below

    sentinel_vals = [9, 10, 99, 998, 999]

    count_cols = ['gravida', 'para', 'miscarriage', 'abortions', 'stillbirths',
                  'order_preg_sudc', 'number_er_visits', 'number_of_hospital_adm']
    for col in [c for c in count_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace(sentinel_vals, np.nan)

    for col in [c for c in ['mom_educ_level', 'dad_educ_level'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([99, 998], np.nan)

    if 'family_income_fa' in df.columns:
        df['family_income_fa'] = pd.to_numeric(
            df['family_income_fa'], errors='coerce').replace([99], np.nan)

    # apgar_at_1_minute / apgar_at_5_minute: value 11 is a data entry error (max valid = 10)
    for col in [c for c in ['apgar_at_1_minute', 'apgar_at_5_minute'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([11], np.nan)

    # abortions -> any_abortions (after sentinel replacement above)
    if 'abortions' in df.columns:
        df['any_abortions'] = df['abortions'].apply(
            lambda x: np.nan if pd.isna(x) else (0.0 if x == 0 else 1.0))
        df.drop(columns='abortions', inplace=True)

    # miscarriage -> any_miscarriage (after sentinel replacement above)
    if 'miscarriage' in df.columns:
        df['any_miscarriage'] = df['miscarriage'].apply(
            lambda x: np.nan if pd.isna(x) else (0.0 if x == 0 else 1.0))
        df.drop(columns='miscarriage', inplace=True)

    # gravida and para: dropped — redundant with order_preg_sudc
    for col in ['gravida', 'para']:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

    # ── 3e. STRING COLUMN PARSING ─────────────────────────────────────────────
    # for col in [c for c in ['bw_percentile', 'bl_percentile'] if c in df.columns]:
    #     df[col] = df[col].apply(parse_percentile)

    if 'sick_visits_per_year' in df.columns:
        df['sick_visits_per_year'] = df['sick_visits_per_year'].apply(parse_sick_visits)

    # ── 3f. FAMILY HISTORY AGGREGATIONS ──────────────────────────────────────
    fam_configs = [
        ('febrile_sz',               range(2, 8), 'any_fam_FS'),
        ('cardiac_arrhythmia',       range(2, 8), 'any_fam_cardiac_arrhythmia'),
        ('syncope',                  range(2, 8), 'any_fam_syncope'),
        ('autism',                   range(6, 8), 'any_fam_autism'),
        ('sudden_cardiac_death',     range(6, 8), 'any_fam_sudden_cardiac_death'),
        ('intellectual_disabilities',range(2, 8), 'any_fam_intellectual_disabilities'),
        ('neuro_other',              range(2, 8), 'any_fam_other_neurological'),
        ('epilepsy',                 range(2, 8), 'any_fam_epilepsy'),
        ('sleep_apnea',              range(2, 8), 'any_fam_sleep_apnea'),
        ('asthma',                   range(2, 8), 'any_fam_asthma'),
        ('sudden_explained_death',   range(2, 8), 'any_fam_sudden_explained_death'),
        ('sudden_unexplained_death', range(2, 8), 'any_fam_sudden_unexplained_death'),
    ]
    for prefix, s_range, new_name in fam_configs:
        df = collapse_family_history(df, prefix, s_range, new_name)

    # ── 3g. REMAINING ONE-HOT CLEANUP ─────────────────────────────────────────
    # Convert any surviving ___cols to numeric and fillna(0)
    remaining_oh = [c for c in df.columns if '___' in c]
    df[remaining_oh] = df[remaining_oh].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Drop any stragglers with sum <= 2 (catches anything missed by step 2a)
    straggler_drop = [c for c in remaining_oh if df[c].sum() <= 2]
    if straggler_drop:
        print(f"  [3g] Straggler low-variance one-hots dropped: {straggler_drop}")
    df.drop(columns=straggler_drop, inplace=True)

    return df


df_recoded = recode_sudc_data(df_dropped, df_raw)
print(f"df_recoded shape: {df_recoded.shape}")
print(f"Missing values remaining: {df_recoded.drop(columns=['sudcrrc_id','child_face_position_fi___1']).isna().sum().sum()}")

df_recoded shape: (317, 187)
Missing values remaining: 1087


## Cell 5 — Imputation + Z-scoring

Column type is inferred dynamically:
- **Binary**: unique non-null values ⊆ {0, 1} → mode imputation
- **Continuous**: >10 unique non-null values → median imputation, then z-score
- **Categorical**: everything else (ordinal scales, counts) → mode imputation

Manual overrides applied before classification:
- `special_needs` dropped (61% missing after recoding; sentinel-dominated)
- `apgar_at_1_minute`, `apgar_at_5_minute` forced into continuous group
  (clinically continuous 0–10 scale; classifier bins them as categorical due to ≤10 unique values)

**Important:** Z-scores are computed on the post-imputation distribution, not
the original observed distribution. This means imputed median values will not
map to exactly z=0 — they map to (median − post_imputation_mean) / post_imputation_std.

In [ ]:
def impute_sudc_data(df_recoded, zscore_params_path=None):
    """
    Impute missing values and z-score continuous/ordinal-treated-as-continuous columns.
    Returns df_imputed plus the three column classification lists.

    Column classification (dynamic, with explicit overrides):
      - binary:      unique non-null values ⊆ {0, 1}          -> mode imputation
      - continuous:  >10 unique non-null values, OR explicitly
                      forced (see FORCE_CONTINUOUS below)       -> median imputation, then z-score
      - categorical: everything else                           -> mode imputation, no z-score

    FORCE_CONTINUOUS overrides the dynamic ≤10-unique-values classifier for
    variables that are ordinal but where a linear/continuous treatment is
    clinically defensible:
      - apgar_at_1_minute, apgar_at_5_minute: clinically continuous 0-10 scale
      - mom_educ_level, dad_educ_level: ordered education brackets, linear
        dose-response assumption standard in clinical literature
      - family_income_fa: ordered income brackets, same reasoning as education
      - order_preg_sudc: birth order -- literal count, equal-interval by construction
      - number_er_visits, number_of_hospital_adm: true counts, strongest case
        for continuous treatment of anything on this list

    Variables intentionally LEFT categorical despite having an ordering:
      - birthlength_precent, birthweight_precent: percentile brackets found to
        have non-equal-width, overlapping ranges (see inspect_ordinal_vs_continuous
        diagnostic) -- a linear assumption is not well supported here
      - general_health_scale: subjective 1-5 Likert rating; clinical convention
        treats these as ordinal-categorical rather than assuming equal psychological
        distance between adjacent levels
    """
    df = df_recoded.copy()

    # ── 4a. Pre-imputation drops ──────────────────────────────────────────────
    # special_needs: 61% missing after recoding (999 sentinel -> NaN).
    if 'special_needs' in df.columns:
        df.drop(columns='special_needs', inplace=True)

    # ── 4b. Classify columns ──────────────────────────────────────────────────
    SKIP_COLS = {'sudcrrc_id', 'child_face_position_fi___1'}

    FORCE_CONTINUOUS = {
        'apgar_at_1_minute', 'apgar_at_5_minute',
        'mom_educ_level', 'dad_educ_level', 'family_income_fa',
        'order_preg_sudc', 'number_er_visits', 'number_of_hospital_adm',
    }

    binary_cols, continuous_cols, categorical_cols = [], [], []

    for col in df.columns:
        if col in SKIP_COLS:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue  # should not exist at this stage

        non_null    = df[col].dropna()
        unique_vals = set(non_null.unique())
        n_unique    = len(unique_vals)

        if col in FORCE_CONTINUOUS:
            continuous_cols.append(col)
        elif unique_vals.issubset({0.0, 1.0}):
            binary_cols.append(col)
        elif n_unique > 10:
            continuous_cols.append(col)
        else:
            categorical_cols.append(col)

    print(f"  Binary cols      : {len(binary_cols)}")
    print(f"  Continuous cols  : {len(continuous_cols)}")
    print(f"  Categorical cols : {len(categorical_cols)}")

    # ── 4c. Impute ────────────────────────────────────────────────────────────
    mode_imputer   = SimpleImputer(strategy='most_frequent')
    median_imputer = SimpleImputer(strategy='median')

    cols = [c for c in binary_cols if df[c].isna().any()]
    if cols: df[cols] = mode_imputer.fit_transform(df[cols])

    cols = [c for c in continuous_cols if df[c].isna().any()]
    if cols: df[cols] = median_imputer.fit_transform(df[cols])

    cols = [c for c in categorical_cols if df[c].isna().any()]
    if cols: df[cols] = mode_imputer.fit_transform(df[cols])

    # ── 4d. Z-score continuous columns ───────────────────────────────────────
    # Applied AFTER imputation. Z-scores use the post-imputation mean/std,
    # so the full column (including previously-missing rows) anchors the scale.
    # Z_SCORE_COLS == continuous_cols by construction now (no separate hardcoded
    # list), so additions to FORCE_CONTINUOUS automatically get z-scored too.
    Z_SCORE_COLS = continuous_cols

    # Capture post-imputation, pre-z-score state for back-transformation later
    df_pre_zscore = df.copy()

    for col in Z_SCORE_COLS:
        std = df[col].std()
        if std == 0:
            print(f"  [WARNING] z-score: '{col}' has zero std after imputation — skipping")
            continue
        df[col] = (df[col] - df[col].mean()) / std

    # ── 4e. Save z-score parameters for later back-transformation ────────────
    # mean/median/std are taken from df_pre_zscore (post-imputation, pre-z-score),
    # so inverse-transforming a z-scored value reproduces exactly what's in
    # df_pre_zscore -- i.e. the imputed value before scaling.
    zscore_params = []
    for col in Z_SCORE_COLS:
        zscore_params.append({
            'variable':  col,
            'mean':      df_pre_zscore[col].mean(),
            'median':    df_pre_zscore[col].median(),
            'std':       df_pre_zscore[col].std(),
            'n_imputed': df_recoded[col].isna().sum(),
        })
    zscore_params_df = pd.DataFrame(zscore_params)

    if zscore_params_path is not None:
        zscore_params_df.to_csv(zscore_params_path, index=False)
        print(f"Saved z-score parameters to: {zscore_params_path}")

    return df, binary_cols, continuous_cols, categorical_cols, zscore_params_df


ZSCORE_PARAMS_PATH = # insert Z score params saving path here

df_imputed, binary_cols, continuous_cols, categorical_cols, zscore_params_df = \
    impute_sudc_data(df_recoded, zscore_params_path=ZSCORE_PARAMS_PATH)

print(f"\ndf_imputed shape: {df_imputed.shape}")
print(f"Missing values remaining: {df_imputed.drop(columns=['sudcrrc_id','child_face_position_fi___1']).isna().sum().sum()}")
print(f"\nContinuous cols (z-scored):")
for c in sorted(continuous_cols): print(f"  {c}")
print(f"\nCategorical cols (mode-imputed, not z-scored):")
for c in sorted(categorical_cols): print(f"  {c}")
print(f"\nZ-score parameters:")
zscore_params_df

  Binary cols      : 166
  Continuous cols  : 15
  Categorical cols : 3
Saved z-score parameters to: /content/drive/MyDrive/S25/Langone/Febrile_Seizure_Prediction/Data/zscore_params.csv

df_imputed shape: (317, 186)
Missing values remaining: 0

Continuous cols (z-scored):
  age_sudc
  apgar_at_1_minute
  apgar_at_5_minute
  dad_educ_level
  days_illness_inyear
  family_income_fa
  father_age
  gestational_age_at_birth
  labor_began
  mom_educ_level
  mother_age
  number_er_visits
  number_of_hospital_adm
  order_preg_sudc
  sick_visits_per_year

Categorical cols (mode-imputed, not z-scored):
  birthlength_precent
  birthweight_precent
  general_health_scale

Z-score parameters:


,variable,mean,median,std,n_imputed
0,age_sudc,37.947634,21.5,44.850380,0
1,labor_began,38.531546,39.0,2.076911,18
2,gestational_age_at_birth,38.684858,39.0,1.805080,11
3,apgar_at_1_minute,8.157729,8.0,1.161232,15
4,apgar_at_5_minute,8.924290,9.0,0.503730,15
5,sick_visits_per_year,3.756025,3.0,4.167703,21
6,number_er_visits,1.170347,1.0,1.261405,11
7,number_of_hospital_adm,0.242902,0.0,0.522580,14
8,mom_educ_level,5.526814,5.0,1.686715,6
9,dad_educ_level,5.034700,5.0,1.856070,16


# Rename map

In [ ]:
RENAME_MAP = {
    # continuous variables
    'age_sudc'                  : 'age_sudc',
    'apgar_at_1_minute'         : 'apgar_1_min',
    'apgar_at_5_minute'         : 'apgar_5_min',
    'days_illness_inyear'       : 'days_illness_inyear',
    'father_age'                : 'father_age',
    'mother_age'                : 'mother_age',
    'gestational_age_at_birth'  : 'gestational_age_at_birth',
    'sick_visits_per_year'      : 'sick_visits_per_year',
    'labor_began'               : 'gestational_week_labor_began',

    # categorical cariables
    'birthlength_precent'       : 'birthlenth_percent',
    'birthweight_precent'       : 'birthweight_percent',
    'dad_educ_level'            : 'dad_educ_level',
    'mom_educ_level'            : 'mom_educ_level',
    'family_income_fa'          : 'family_income_fa',
    'general_health_scale'      : 'general_health_scale',
    'number_er_visits'          : 'num_er_visits',
    'number_of_hospital_adm'    : 'num_hospital_adm',
    'order_preg_sudc'           : 'order_preg_sudc',

    # binary variables
    'pmh_pregnancy___2'         : 'hypertension_during_pregnancy',
    'pmh_pregnancy___3'         : 'tocolytics_given_during_pregnancy',
    'pmh_pregnancy___4'         : 'proteinuria_during_pregnancy',
    'pmh_pregnancy___6'         : 'bedrest_during_pregnancy',
    'pmh_pregnancy___7'         : 'diabetes_req_insulin_during_pregnancy',
    'pmh_pregnancy___8'         : 'diabetes_not_req_insulin_during_pregnancy',
    'pmh_pregnancy___9'         : 'vaginal_bleeding_in_1st_trimester_during_pregnancy',
    'pmh_pregnancy___10'        : 'vaginal_bleeding_in_2nd_trimester_during_pregnancy',
    'pmh_pregnancy___11'        : 'vaginal_bleeding_in_3rd_trimester_during_pregnancy',
    'pmh_pregnancy___13'        : 'cigarette_smoking_during_pregnancy',
    'pmh_pregnancy___15'        : 'vitamin_during_pregnancy',
    'pmh_pregnancy___16'        : 'iron_supplements_during_pregnancy',
    'pmh_pregnancy___17'        : 'aspirin_during_pregnancy',
    'pmh_pregnancy___18'        : 'acetominophen_during_pregnancy',
    'pmh_pregnancy___19'        : 'ibuprofen_during_pregnancy',
    'pmh_pregnancy___23'        : 'antacid_medication_during_pregnancy',
    'pmh_pregnancy___24'        : 'blood_pressure_medication_during_pregnancy',
    'pmh_pregnancy___26'        : 'antidepressants_during_pregnancy',
    'pmh_pregnancy___27'        : 'decongestants_during_pregnancy',
    'pmh_pregnancy___28'        : 'THC_during_pregnancy',
    'pmh_pregnancy___30'        : 'antibiotics_during_pregnancy',
    'pmh_pregnancy___31'        : 'maternal_infection_during_pregnancy',
    'pmh_pregnancy___32'        : 'gestational_hormones',
    'pmh_pregnancy___33'        : 'hormones_during_pregnancy',
    'pmh_pregnancy___34'        : 'illicit_drug_use_during_pregnancy',
    'pmh_pregnancy___35'        : 'premature_labor_history',
    'pmh_pregnancy___38'        : 'third_trimester_fever',
    'pmh_pregnancy___39'        : 'preeclampsia_eclampsia_history',
    'labor_details___1'         : 'spontaneous_labor',
    'labor_details___2'         : 'used_drugs_to_initiate_labor',
    'labor_details___3'         : 'used_drugs_to_strengthen_contractions',
    'labor_details___4'         : 'child_in_distress_during_labor',
    'labor_details___5'         : 'meconium_staining_of_amniotic_fluid_present',
    'labor_details___6'         : 'umbilical_cord_problems',
    'labor_details___11'        : 'child_in_distress_upon_delivery',
    'in_hospital_newborn_hx___1': 'in_hospital_newborn_anemic',
    'in_hospital_newborn_hx___2': 'in_hospital_newborn_req_oxygen',
    'in_hospital_newborn_hx___4': 'in_hospital_newborn_intubation',
    'in_hospital_newborn_hx___5': 'in_hospital_newborn_antibiotics',
    'in_hospital_newborn_hx___6': 'in_hospital_newborn_jaundice',
    'in_hospital_newborn_hx___7': 'in_hospital_newborn_req_surgery',
    'in_hospital_newborn_hx___10':'in_hospital_newborn_tachycardia',
    'in_hospital_newborn_hx___13':'in_hospital_newborn_apnea',
    'in_hospital_newborn_hx___14':'in_hospital_newborn_cyanosis',
    'in_hospital_newborn_hx___16':'in_hospital_newborn_respiratory_distress',
    'in_hospital_newborn_hx___18':'in_hospital_newborn_fever',
    'in_hospital_newborn_hx___21':'in_hospital_newborn_hypoglycemia',
    'in_hospital_newborn_hx___22':'in_hospital_newborn_NICU',
    'in_hospital_newborn_hx___23':'in_hospital_newborn_discharge_home_w/_mom',
    'in_hospital_newborn_hx___24':'in_hospital_newborn_extended_hospital_stay',
    'development_wnl'           : 'developmental_milestones_normal',
    'neuro_condit___4'          : 'child_autism_MH',
    'neuro_condit___5'          : 'child_intellectual_disability_MH',
    'cardiaccondit___8'         : 'child_Atrial_Septal_Defect_MH',
    'cardiaccondit___9'         : 'child_Ventricular_Septal_Defect_MH',
    'respcondit___1'            : 'child_asthma_MH',
    'respcondit___4'            : 'child_reactive_airway_disease_MH',
    'respcondit___5'            : 'child_RSV_MH',
    'psychcondit___1'           : 'child_anxiety_MH',
    'psychcondit___4'           : 'child_depression_MH',
    'psychcondit___6'           : 'child_ADD/ADHD_MH',
    'stillbirths'               : 'any_stillbirths',
    'fertility_meds___1'        : 'fertility_meds_IVF',
    'fertility_meds___2'        : 'fertility_meds_IUI',
    'fertility_meds___3'        : 'fertility_meds_injectable',
    'fertility_meds___5'        : 'fertility_meds_oral',
    'preg_complications'        : 'any_preg_complications',
    'alcohol'                   : 'any_alchol_preg',
    'caffeine_coffee_tea'       : 'any_caffeine_coffee_tea',
    'cocaine'                   : 'any_cocaine',
    'amphetamines'              : 'any_amphetamines',
    'opiods_heroin'             : 'any_opioids_heroin',
    'delivery_complications'    : 'any_delivery_complications',
    'stop_breathing'            : 'if_stop_breathing',
    'bradycardia'               : 'if_bradycardia',
    'require_cpr'               : 'if_req_cpr',
    'seizure'                   : 'if_have_seizure',
    'sudc_health_hx___3'        : 'sudc_child_h/o_limpness',
    'sudc_health_hx___4'        : 'sudc_child_h/o_blue',
    'sudc_health_hx___5'        : 'sudc_child_h/o_choking',
    'sudc_health_hx___7'        : 'sudc_child_h/o_>12hr_w/o_eating',
    'sudc_health_hx___8'        : 'sudc_child_h/o_>12hr_vomiting',
    'sudc_health_hx___9'        : 'sudc_child_h/o_breathing_problems',
    'sudc_health_hx___10'       : 'sudc_child_h/o_heart_problems',
    'sudc_health_hx___12'       : 'sudc_child_h/o_GER',
    'sudc_health_hx___13'       : 'sudc_child_use_apnea_monitor',
    'sudc_health_hx___15'       : 'sudc_child_hearing_impairment',
    'sudc_health_hx___16'       : 'sudc_child_speech_delay',
    'sudc_health_hx___17'       : 'sudc_child_vision_impairment',
    'sudc_health_hx___18'       : 'sudc_child_h/o_wheezing',
    'sudc_health_hx___20'       : 'sudc_child_h/o_fainting',
    'expressive_speech'         : 'expressive_speech_ok',
    'comprehension'             : 'comprehension_ok',
    'fine_motor'                : 'fine_motor_ok',
    'gross_motor'               : 'gross_motor_ok',
    'behavior'                  : 'behavior_ok',
    'rolled'                    : 'rolled_<5m',
    'crawled'                   : 'crawled_<10m',
    'started'                   : 'started_walking_<17m',
    'first_words'               : 'first_words_<14m',
    'allergy_hx'                : 'have_allergy',
    'recent_actvity_hx___1'     : 'recent_head_trauma',
    'recent_actvity_hx___2'     : 'recent_flying_on_airplane',
    'recent_actvity_hx___3'     : 'recent_visit_ER',
    'recent_actvity_hx___5'     : 'recent_visit_doctor',
    'recent_actvity_hx___6'     : 'recent_exposure_contagious_disease',
    'recent_actvity_hx___7'     : 'recent_suffer_illness',
    'last_48hr_sxs___2'         : 'last_48hr_cold_symptoms',
    'last_48hr_sxs___3'         : 'last_48hr_lethargy',
    'last_48hr_sxs___4'         : 'last_48hr_crankiness',
    'last_48hr_sxs___5'         : 'last_48hr_excessive_crying',
    'last_48hr_sxs___6'         : 'last_48hr_appetite_changes',
    'last_48hr_sxs___7'         : 'last_48hr_vomiting',
    'last_48hr_sxs___9'         : 'last_48hr_fever',
    'last_48hr_sxs___11'        : 'last_48hr_diarrhea',
    'last_48hr_sxs___12'        : 'last_48hr_other_stool_changes',
    'seizures_were_categorized___1':'simple_fs',
    'seizures_were_categorized___2':'complex_fs',
    'parasomnias___2'           : 'talked_during_sleep_<6m_prior',
    'parasomnias___3'           : 'restless_during_sleep_<6m_prior',
    'parasomnias___4'           : 'sleepwalked_<6m_prior',
    'parasomnias___5'           : 'grinded_teeth_<6m_prior',
    'parasomnias___6'           : 'night_terror_<6m_prior',
    'parasomnias___10'          : 'awoke_sweating_<6m_prior',
    'parasomnias___7'           : 'scary_dreams_<6m_prior',
    'awokeoften'                : 'awoke_often_>2/week',
    'snoring'                   : 'snoring',
    'fallingasleep'             : 'difficulty_falling_asleep',
    'stayingasleep'             : 'difficulty_staying_asleep',
    'favesleep'                 : 'have_favorite_sleep_position',
}

In [ ]:
def apply_rename_map(df, rename_map):
    df = df.copy()
    valid_renames = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=valid_renames)

    return df


df_final = apply_rename_map(df_imputed, RENAME_MAP)
print(f"df_final shape: {df_final.shape}")

df_final shape: (317, 186)


# Cell 6 — Save

In [ ]:
df_final.to_csv(SAVE_PATH, index=False)
print(f"Saved to: {SAVE_PATH}")
print(f"Final shape: {df_final.shape}")
print(f"\nColumn list ({df_final.shape[1]} cols):")

Saved to: /content/drive/MyDrive/S25/Langone/Febrile_Seizure_Prediction/Data/DF_IMPUTED_JUNE.csv
Final shape: (317, 186)

Column list (186 cols):
